# Лабораторная работа 11. Online softmax и FlashAttention (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 4:** инференс и эффективность

**Цель.** Разобрать алгоритм точного attention, который не хранит матрицу \(T\times T\) и потому быстрее наивной реализации на GPU.

**Что нужно сдать:**
1. Online softmax и блочное attention на PyTorch, численно эквивалентные обычному (FP32, FP16, BF16).
2. Прямой проход FlashAttention на Triton (каузальная маска, тайлинг по Q и K/V).
3. Бенчмарк скорости и пиковой памяти: наивная реализация, блочная, `scaled_dot_product_attention`, Triton.
4. Roofline-оценку для T4 и объяснение результатов.
5. Дополнительно: обратный проход с пересчётом и проверкой градиентов.

**Вычисления.** Пункты 2–3 с Triton требуют GPU (Colab T4). На CPU ноутбук выполняет всё остальное и пропускает ячейки Triton.

**Теория. Online softmax.** Для блоков \(x^{(1)}, x^{(2)}, \dots\) поддерживаются текущий максимум \(m\) и нормировка \(\ell\):
\[
m^{new} = \max(m, \max_j x^{(b)}_j),\qquad
\ell^{new} = e^{m - m^{new}}\ell + \sum_j e^{x^{(b)}_j - m^{new}} .
\]
По индукции \(\ell = \sum_{j \le \text{текущий}} e^{x_j - m}\), поэтому в конце \(\mathrm{softmax}(x)_j = e^{x_j - m}/\ell\). Для attention вместе с \(\ell\) так же пересчитывается ненормированный выход:
\[
o^{new} = e^{m - m^{new}}\,o + \sum_j e^{s_j - m^{new}}\,v_j,\qquad O = o/\ell .
\]

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

## 1. Online softmax для вектора

In [ ]:
def online_softmax(x, block=16):
    """Двухпроходный алгоритм: (1) потоковые m и l по блокам; (2) нормировка."""
    # TODO: ваш код
    raise NotImplementedError

x = torch.randn(1000) * 20
assert torch.allclose(online_softmax(x), x.softmax(0), atol=1e-6)
x = torch.tensor([1000.0, 1001.0, -1000.0])                     # устойчивость к переполнению
assert torch.allclose(online_softmax(x, block=1), x.softmax(0))
print("online softmax совпадает с torch.softmax")

## 2. Блочное attention (FlashAttention-подобный прямой проход на PyTorch)

Входы `q, k, v` формы `(B, H, T, d)`. Внешний цикл — по блокам запросов размера `Br`, внутренний — по блокам ключей `Bc`. Аккумуляторы \(m, \ell, o\) держите в FP32 независимо от типа входа. При каузальной маске пропускайте блоки ключей, целиком лежащие правее блока запросов. Функция дополнительно возвращает logsumexp \(L = m + \log\ell\) — он понадобится для обратного прохода.

In [ ]:
def naive_attention(q, k, v, causal=False):
    s = (q @ k.transpose(-1, -2)).float() / math.sqrt(q.size(-1))
    if causal: s = s.masked_fill(torch.ones(s.shape[-2:], dtype=torch.bool, device=q.device).triu(1), float("-inf"))
    return (s.softmax(-1) @ v.float()).to(q.dtype)

def blocked_attention(q, k, v, causal=False, Br=64, Bc=64):
    B, H, T, d = q.shape; scale = 1 / math.sqrt(d)
    O = torch.empty_like(q); LSE = torch.empty(B, H, T, dtype=torch.float32, device=q.device)
    # TODO: ваш код
    raise NotImplementedError
    return O, LSE

torch.manual_seed(0)
for dtype, tol in [(torch.float32, 1e-5), (torch.bfloat16, 2e-2), (torch.float16, 5e-3)]:
    q, k, v = (torch.randn(2, 3, 200, 64).to(dtype) for _ in range(3))
    for causal in (False, True):
        ref = F.scaled_dot_product_attention(q.float(), k.float(), v.float(), is_causal=causal)
        out, _ = blocked_attention(q, k, v, causal, Br=48, Bc=32)
        err = (out.float() - ref).abs().max().item()
        print(f"{str(dtype):15s} causal={causal}: max|Δ| = {err:.2e}"); assert err < tol

## 3. FlashAttention на Triton (GPU)

Реализуйте ядро прямого прохода. Каждая программа обрабатывает один блок из `BLOCK_M` запросов для одной пары (batch, head):
1. загрузить блок \(Q\);
2. в цикле по блокам \(K, V\) размера `BLOCK_N` вычислить \(S = QK^\top\cdot\text{scale}\), применить маски (выход за длину и каузальную), обновить \(m, \ell\), аккумулятор;
3. записать \(O = \text{acc}/\ell\).

Для каузальной маски цикл идёт только до блока, содержащего диагональ.

In [ ]:
HAS_TRITON = False
if DEVICE == "cuda":
    try:
        import triton, triton.language as tl
        HAS_TRITON = True
    except ImportError:
        os.system("pip -q install triton"); import triton, triton.language as tl; HAS_TRITON = True
print("Triton доступен:", HAS_TRITON)

In [ ]:
if HAS_TRITON:
    @triton.jit
    def flash_fwd_kernel(Q, K, V, O, sm_scale, T, stride_bh, stride_t,
                         BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, D: tl.constexpr, CAUSAL: tl.constexpr):
        pid_m = tl.program_id(0); pid_bh = tl.program_id(1)
        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_d = tl.arange(0, D)
        base = pid_bh * stride_bh
        # TODO: ваш код
        raise NotImplementedError

    def flash_attention_triton(q, k, v, causal=False, BLOCK_M=64, BLOCK_N=64):
        B, H, T, D = q.shape
        assert D in (16, 32, 64, 128) and q.dtype in (torch.float16, torch.bfloat16)
        q3, k3, v3 = (t.contiguous().view(B * H, T, D) for t in (q, k, v)); o = torch.empty_like(q3)
        grid = (triton.cdiv(T, BLOCK_M), B * H)
        flash_fwd_kernel[grid](q3, k3, v3, o, 1 / math.sqrt(D), T, T * D, D, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, D=D, CAUSAL=causal)
        return o.view(B, H, T, D)

    for causal in (False, True):
        q, k, v = (torch.randn(2, 4, 1000, 64, device="cuda", dtype=torch.float16) for _ in range(3))
        ref = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
        err = (flash_attention_triton(q, k, v, causal) - ref).abs().max().item()
        print(f"Triton causal={causal}: max|Δ| = {err:.2e}"); assert err < 1e-2
else:
    print("GPU нет — ячейка Triton пропущена. Запустите ноутбук в Colab с GPU T4.")

## 4. Бенчмарк скорости и памяти

Для длин \(T\) от 512 до 8192 (на CPU — до 2048), \(d = 64\), \(B\cdot H = 8\) измерьте время прямого прохода и пиковую память (на GPU — `torch.cuda.max_memory_allocated`; на CPU — оценка по размеру промежуточных тензоров) для:
- наивной реализации;
- блочной PyTorch-реализации;
- `F.scaled_dot_product_attention`;
- Triton-ядра (на GPU).

Объясните, почему блочная реализация на PyTorch медленнее наивной, хотя использует тот же алгоритм, что и FlashAttention.

In [ ]:
def bench(fn, reps=3):
    if DEVICE == "cuda": torch.cuda.synchronize()
    fn()
    if DEVICE == "cuda": torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(reps): fn()
    if DEVICE == "cuda": torch.cuda.synchronize()
    mem = torch.cuda.max_memory_allocated() / 2 ** 20 if DEVICE == "cuda" else float("nan")
    return (time.time() - t0) / reps * 1000, mem

LENGTHS = [256, 512, 1024, 2048] if DEVICE == "cpu" else [512, 1024, 2048, 4096, 8192]
dtype = torch.float16 if DEVICE == "cuda" else torch.float32
rows = []
# TODO: ваш код (заполните rows для всех методов и длин; для CPU добавьте оценку памяти naive = B*H*T*T*4 байт)

## 5. Roofline-оценка для T4

Параметры T4: пиковая производительность FP16 на тензорных ядрах около 65 TFLOP/s, пропускная способность памяти около 320 ГБ/с. Для одной головы с длиной \(T\) и \(d = 64\) в FP16 оцените:
1. FLOPs прямого прохода (\(4T^2d\) без учёта маски);
2. объём обмена с HBM для наивной реализации (чтение \(Q, K, V\), запись и чтение \(S\) и \(P\), запись \(O\));
3. то же для FlashAttention (\(Q\) и \(O\) — один раз, \(K, V\) — \(T/B_r\) раз);
4. арифметическую интенсивность и ограничение (compute-bound или memory-bound) в обоих случаях, сравнив с точкой перегиба \(65\cdot10^{12} / 320\cdot10^{9} \approx 200\) FLOP/байт.

In [ ]:
def roofline(T, d=64, Br=64, bytes_per=2, peak=65e12, bw=320e9):
    # TODO: ваш код
    raise NotImplementedError

for T in (1024, 4096, 16384):
    print(T, {k: {kk: (round(vv, 1) if isinstance(vv, float) else vv) for kk, vv in v.items()} for k, v in roofline(T).items()})

## 6. Дополнительно: обратный проход с пересчётом

Реализуйте `torch.autograd.Function`, где `forward` вызывает `blocked_attention` и сохраняет только \(Q, K, V, O\) и \(L\) (без \(P\)). В `backward` для каждого блока пересчитайте \(P_{ij} = \exp(S_{ij} - L_i)\) и накопите
\[
dV_j \mathrel{+}= P_{ij}^\top dO_i,\quad dP_{ij} = dO_i V_j^\top,\quad dS_{ij} = P_{ij}\odot(dP_{ij} - D_i),\quad
dQ_i \mathrel{+}= \text{scale}\cdot dS_{ij}K_j,\quad dK_j \mathrel{+}= \text{scale}\cdot dS_{ij}^\top Q_i,
\]
где \(D_i = \mathrm{rowsum}(dO_i \odot O_i)\). Сверьте градиенты с autograd.

In [ ]:
class FlashAttnFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, causal):
        O, L = blocked_attention(q, k, v, causal)
        ctx.save_for_backward(q, k, v, O, L); ctx.causal = causal
        return O
    @staticmethod
    def backward(ctx, dO):
        q, k, v, O, L = ctx.saved_tensors; causal = ctx.causal
        B, H, T, d = q.shape; scale = 1 / math.sqrt(d); Br = Bc = 64
        dq, dk, dv = torch.zeros_like(q), torch.zeros_like(k), torch.zeros_like(v)
        Dvec = (dO * O).sum(-1)
        # TODO: ваш код
        raise NotImplementedError
        return dq, dk, dv, None

q, k, v = (torch.randn(1, 2, 150, 32, dtype=torch.float64, requires_grad=True) for _ in range(3))
for causal in (False, True):
    dO = torch.randn(1, 2, 150, 32, dtype=torch.float64)
    g1 = torch.autograd.grad((FlashAttnFn.apply(q, k, v, causal) * dO).sum(), (q, k, v))
    g2 = torch.autograd.grad((naive_attention(q, k, v, causal) * dO).sum(), (q, k, v))
    print(f"causal={causal}:", [f"{(a - b).abs().max().item():.1e}" for a, b in zip(g1, g2)])
    assert all(torch.allclose(a, b, atol=1e-5) for a, b in zip(g1, g2))   # прямой проход накапливает в FP32

## Вопросы

1. Докажите корректность online softmax по индукции. Почему вычитание максимума не меняет результат, но необходимо численно?
2. Почему FlashAttention быстрее наивной реализации, хотя выполняет **больше** FLOPs в обратном проходе (пересчёт \(P\))?
3. Как размер блоков `BLOCK_M`, `BLOCK_N` ограничен размером SRAM (shared memory) и чем грозит слишком большой блок?
4. Почему блочная реализация на PyTorch в п. 4 медленнее наивной? Что именно даёт слияние операций в одно ядро?
5. Как изменится анализ roofline на фазе decode (один запрос, длинный кэш)? Какие оптимизации применяются там (FlashDecoding, split-K)?